# Badge 1: Data Warehousing Workshop: Lesson 11: Intro to Semi-Structured Data

AUTHOR_INGEST_JSON, a JSON file format, and querying VARIANT data.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, which are swapped for `BADGE1_DWW_ADMIN` (ACCOUNTADMIN), `BADGE1_DWW_ROLE` (SYSADMIN) and `BADGE1_DWW_WH` (COMPUTE_WH).
- Cells marked **SQL equivalent** replace a UI-only step; **Derived solution** marks challenge-lab answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on the `@UTIL_DB.PUBLIC.BADGE1_DOWNLOADS` stage, which stands in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE1_DWW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE1_DWW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE1_DWW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{ADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse();

## Page: 🎭 Semi-Structured Data Formats

Video and knowledge checks only.

## Page: 🎯 Create JSON Table & File Format, and Load the File

> **Course:** 🎯 Download a JSON File (author_with_header.json). Notice the square bracket that surrounds the author records; it could cause all the data to end up in a single row.

In [ ]:
%%sql -r create_author_ingest_json
// JSON DDL Scripts
use database library_card_catalog;
use role {{SYSADMIN_ROLE}};

// Create an Ingestion Table for JSON Data
create table library_card_catalog.public.author_ingest_json
(
  raw_author variant
);

> **Course:** 🎯 Create a File Format to Load the JSON Data: replace each <?> with TRUE or FALSE. The default for each setting is FALSE. To load the file data into separate rows, one of the values will need to be set to TRUE.

Course template:

```sql
//Create File Format for JSON Data 
create file format library_card_catalog.public.json_file_format
type = 'JSON' 
compression = 'AUTO' 
enable_octal = <?>
allow_duplicate = <?> 
strip_outer_array = <?>
strip_null_values = <?> 
ignore_utf8_errors = <?>; 
```

In [ ]:
%%sql -r create_json_ff
-- Derived solution: STRIP_OUTER_ARRAY = TRUE splits the bracketed array into rows
create file format library_card_catalog.public.json_file_format
type = 'JSON' 
compression = 'AUTO' 
enable_octal = FALSE
allow_duplicate = FALSE 
strip_outer_array = TRUE
strip_null_values = FALSE 
ignore_utf8_errors = FALSE; 

> **Course:** 🎯 Load the data into the new table using the file format you created. Test the file format first by querying the file (DWW_165). Stage: @util_db.public.my_internal_stage.

In [ ]:
%%sql -r put_author_json
-- SQL equivalent of uploading the downloaded file(s) through the stage's + Files dialog
use role {{ADMIN_ROLE}};
copy files into @util_db.public.my_internal_stage
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('author_with_header.json');
list @util_db.public.my_internal_stage;

> ⚠️ **Audit note:** The JSON table is owned by SYSADMIN but the stage by ACCOUNTADMIN; the test query and COPY INTO run as the ACCOUNTADMIN stand-in.

In [ ]:
%%sql -r test_json_ff
use role {{ADMIN_ROLE}};
select $1
from @util_db.public.my_internal_stage/author_with_header.json
(file_format => library_card_catalog.public.json_file_format);

In [ ]:
%%sql -r copy_author_json
-- Derived solution
copy into library_card_catalog.public.author_ingest_json
from @util_db.public.my_internal_stage
files = ('author_with_header.json')
file_format = (format_name = library_card_catalog.public.json_file_format);

> **Course:** 🎯 View the JSON Rows (DWW_166).

In [ ]:
%%sql -r view_author_json
select * from library_card_catalog.public.author_ingest_json;

## Page: 🥋 Query the JSON Table

In [ ]:
%%sql -r query_author_json
use database library_card_catalog;
use schema public;
//returns AUTHOR_UID value from top-level object's attribute
select raw_author:AUTHOR_UID
from author_ingest_json;

//returns the data in a way that makes it look like a normalized table
SELECT 
 raw_author:AUTHOR_UID
,raw_author:FIRST_NAME::STRING as FIRST_NAME
,raw_author:MIDDLE_NAME::STRING as MIDDLE_NAME
,raw_author:LAST_NAME::STRING as LAST_NAME
FROM AUTHOR_INGEST_JSON;

### 🤖 DORA check DWW16 goes here
The course has the learner run **DWW16** (Check number of rows, expects 6) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW16**, as a local check plus the exact DORA code.